# 01. 브라질 이커머스(Olist) 데이터 분석 ① — 데이터 이해 · 고객 · 매출

브라질 온라인 쇼핑몰 **Olist** 의 실제 주문 데이터(약 10만 건)로 **탐색적 데이터 분석(EDA)** 의 전 과정을 따라갑니다.
이 시리즈는 노트북 3개로 이루어져 있습니다.

| 노트북 | 내용 | 결과물 |
|---|---|---|
| **01 (지금)** | 데이터 구조 이해 → 고객 분석 → 매출 분석 | 분석 노트 |
| 02 | 시간 패턴 → 카테고리 → 배송과 리뷰 | 분석 노트 + 인사이트 정리 |
| 03 | 분석 결과를 **웹 대시보드(Streamlit)** 로 만들고 **GitHub → Streamlit Cloud** 로 배포 | 누구나 접속하는 웹 서비스 |

<div style="border:1px solid #9ABBE0;border-left:5px solid #1F3A7A;background:#EAF2FD;padding:10px 14px;border-radius:4px">

**이 노트북의 학습 목표**

1. 여러 개의 CSV 테이블이 **어떤 열(키)로 연결되는지** 이해한다.
2. `value_counts` · `nunique` · `groupby` · `merge` 로 고객과 결제 데이터를 요약한다.
3. **분석 대상(어떤 주문을 매출로 볼 것인가)** 을 먼저 정의하고, 날짜 데이터를 다뤄 월별 매출을 구한다.
4. plotly 로 보기 좋은 그래프를 만들고 숫자에서 **인사이트**를 읽어 낸다.

</div>

In [1]:
# ── 준비: 라이브러리 불러오기 ──────────────────────────────
import pandas as pd                    # 표(DataFrame) 다루기
import plotly.express as px            # 짧은 코드로 그래프 그리기
import plotly.graph_objects as go      # 그래프를 세밀하게 꾸미기
import plotly.io as pio

# 그래프 표시 방식: VS Code · JupyterLab · Colab 어디서나 보이도록 설정
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"  # 흰 배경 테마를 기본으로

# 데이터 폴더 위치 — 이 노트북 폴더의 '한 단계 위(..)'에 있는 00_data 폴더
PATH = "../00_data/"

pd.set_option("display.max_columns", 30)       # 열이 많아도 생략 없이 보이게
pd.set_option("display.float_format", "{:,.2f}".format)   # 실수는 천 단위 쉼표 + 소수 둘째 자리
print("pandas", pd.__version__)

pandas 3.0.6


---
# PART A. 데이터 이해하기

## A-1. Olist 는 어떤 회사인가?

Olist 는 브라질의 **온라인 마켓플레이스 연결 서비스**입니다. 작은 판매자(셀러)가 Olist 와 계약하면 여러 대형 쇼핑몰에 상품을 올려 팔 수 있습니다.

```
 판매자가 상품 등록 → 고객이 주문·결제 → 판매자가 물류사로 발송 → 고객이 상품 수령 → 고객에게 만족도 설문(리뷰) 발송
```

- **데이터 출처** : Kaggle 「Brazilian E-Commerce Public Dataset by Olist」 (라이선스 CC BY-NC-SA 4.0 — 비영리 · 출처 표시 조건으로 공개)
- **기간** : 2016년 9월 ~ 2018년 10월, 주문 약 10만 건
- **금액 단위** : 브라질 헤알(BRL). 그래프와 코드 출력에서는 브라질식 표기 `R$` 를 씁니다.
- 개인 정보 보호를 위해 고객·판매자 ID 는 암호처럼 바뀌어(익명화) 있습니다.

## A-2. 분석 질문 만들기

좋은 분석은 **질문**에서 시작합니다. 이 시리즈에서 답할 질문은 다음과 같습니다.

| 번호 | 질문 | 노트북 |
|---|---|---|
| Q1 | 고객은 몇 명이고, **다시 사는 고객**은 얼마나 될까? | 01 |
| Q2 | 고객은 **어느 지역**에 많을까? | 01 |
| Q3 | 고객은 주로 **어떤 결제 수단**을 쓸까? | 01 |
| Q4 | 주문 1건의 **평균 금액**과 **월별 매출**은? | 01 |
| Q5 | 언제(연·월·요일·시간) 많이 살까? | 02 |
| Q6 | 어떤 **카테고리**가 많이 팔리고 돈이 될까? | 02 |
| Q7 | 배송은 며칠 걸리고, **배송이 리뷰 점수에 영향을** 줄까? | 02 |

<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**EDA 체크리스트** — 열의 종류에 따라 확인하는 방법이 다릅니다.

- **수치형**(가격, 배송일 등) : `describe()` 로 최솟값 · 사분위수 · 중앙값 · 평균 · 최댓값 확인 → 이상치가 있으면 상자 그림(box) · 히스토그램
- **범주형**(결제 수단, 지역 등) : `value_counts()` 로 종류별 개수 → 막대 그래프(개수) · 원 그래프(비율)
- **시계열**(주문 날짜) : 날짜형으로 바꾼 뒤 기간별로 묶기 → 선 · 막대 그래프
- **두 변수의 관계** : 교차표 · 산점도 · 히트맵

</div>

## A-3. 테이블 9개 한꺼번에 불러오기

같은 일을 9번 반복하는 대신 **딕셔너리 + for 반복문**으로 불러와 크기를 한 번에 비교합니다.

In [2]:
# 파일 이름표: {짧은 이름: 실제 파일 이름}
files = {
    "orders":    "olist_orders_dataset.csv",            # 주문 (한 행 = 주문 1건)
    "customers": "olist_customers_dataset.csv",         # 고객 (한 행 = 주문할 때의 고객 정보)
    "items":     "olist_order_items_dataset.csv",       # 주문 상품 (한 행 = 주문에 담긴 상품 1개)
    "payments":  "olist_order_payments_dataset.csv",    # 결제 (한 행 = 결제 1번)
    "reviews":   "olist_order_reviews_dataset.csv",     # 리뷰 (한 행 = 리뷰 1개)
    "products":  "olist_products_dataset.csv",          # 상품 정보
    "sellers":   "olist_sellers_dataset.csv",           # 판매자 정보
    "category":  "product_category_name_translation.csv",  # 카테고리 이름 (포르투갈어 → 영어)
    "geo":       "olist_geolocation_dataset.csv",       # 우편번호별 위도·경도
}

tables = {}                                     # 불러온 표를 담을 빈 딕셔너리
for name, file in files.items():                # 이름과 파일을 하나씩 꺼내며 반복
    tables[name] = pd.read_csv(PATH + file)     # CSV → DataFrame

# 표마다 행 수 · 열 수 · 빈 칸(결측치) 수를 정리
summary = pd.DataFrame({
    "행 수": {k: v.shape[0] for k, v in tables.items()},
    "열 수": {k: v.shape[1] for k, v in tables.items()},
    "빈 칸 수": {k: int(v.isna().sum().sum()) for k, v in tables.items()},
})
summary

,행 수,열 수,빈 칸 수
orders,99441,8,4908
customers,99441,5,0
items,112650,7,0
payments,103886,5,0
reviews,100000,7,146532
products,32951,9,2448
sellers,3095,4,0
category,71,2,0
geo,1000163,5,0


In [3]:
# 자주 쓸 표는 짧은 변수 이름으로 꺼내 둡니다.
orders, customers, items, payments = tables["orders"], tables["customers"], tables["items"], tables["payments"]
reviews, products, category = tables["reviews"], tables["products"], tables["category"]

## A-4. 테이블은 어떻게 연결될까? (키 이해하기)

여러 표를 합치려면(`merge`) **어떤 열이 연결 고리(키)** 인지 알아야 합니다.

```
 customers ──customer_id──▶ orders ◀──order_id── payments   (주문 1건에 결제 여러 번 가능)
                              │  ▲
                   order_id   │  └──order_id── reviews      (주문 1건에 리뷰 보통 1개)
                              ▼
                            items ──product_id──▶ products ──category_name──▶ category
                              │
                              └────seller_id────▶ sellers
```

| 키 | 의미 | 주의할 점 |
|---|---|---|
| `order_id` | 주문 번호 | orders 에서는 한 번씩, items · payments 에서는 **여러 번** 나올 수 있음 |
| `customer_id` | **주문할 때마다** 새로 생기는 고객 번호 | 같은 사람이라도 주문마다 다름 |
| `customer_unique_id` | **실제 사람 한 명**을 나타내는 번호 | 고객 수를 셀 때는 이것을 사용 |
| `product_id` / `seller_id` | 상품 · 판매자 번호 | |

In [4]:
# 키가 '한 번씩만' 나오는지(is_unique) 확인 — merge 결과 행 수를 예상하는 데 꼭 필요합니다.
print("orders.order_id 중복 없음?      ", orders["order_id"].is_unique)
print("orders.customer_id 중복 없음?   ", orders["customer_id"].is_unique, " ← 주문마다 customer_id 가 새로 생김")

# 주문 1건에 상품 · 결제가 몇 개씩 붙어 있는지
items_per_order = items.groupby("order_id").size()        # size(): 그룹마다 행 개수
pays_per_order = payments.groupby("order_id").size()
print("주문 1건의 최대 상품 수:", items_per_order.max(), "/ 상품 2개 이상 주문 비율:", round((items_per_order > 1).mean() * 100, 1), "%")
print("주문 1건의 최대 결제 횟수:", pays_per_order.max(), "/ 결제 2번 이상 주문 수:", (pays_per_order > 1).sum())

orders.order_id 중복 없음?       True
orders.customer_id 중복 없음?    True  ← 주문마다 customer_id 가 새로 생김
주문 1건의 최대 상품 수: 21 / 상품 2개 이상 주문 비율: 9.9 %
주문 1건의 최대 결제 횟수: 29 / 결제 2번 이상 주문 수: 2961


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**기억하기** — orders 와 payments 를 그냥 `merge` 하면 결제를 여러 번 한 주문은 **행이 여러 개로 늘어납니다.**
그 상태에서 행 수를 세면 '주문 수'가 아니라 '결제 수'가 됩니다. 그래서 payments 는 **주문별로 먼저 합친 뒤**(`groupby`) merge 합니다(PART C).

</div>

## A-5. orders 테이블 자세히 보기

In [5]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [6]:
orders.info()   # 날짜 열이 모두 str(문자)라는 점, 일부 날짜 열에 빈 칸이 있다는 점을 확인

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


In [7]:
# 주문 상태(order_status)별 개수 — delivered(배송 완료)가 대부분
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

**빈 칸은 왜 생겼을까?** 배송 완료일(`order_delivered_customer_date`)이 비어 있는 주문을 상태별로 세어 봅니다.
`pd.crosstab(행, 열)` 은 두 열의 조합별 개수를 세는 **교차표**를 만듭니다.

In [8]:
pd.crosstab(orders["order_status"], orders["order_delivered_customer_date"].isna(),
            rownames=["주문 상태"], colnames=["배송완료일이 비어 있음?"])

배송완료일이 비어 있음?,False,True
주문 상태,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**해석** — 배송 완료일이 빈 주문은 대부분 아직 배송 중(shipped)이거나 취소(canceled) · 품절(unavailable)된 주문입니다. 즉, 빈 칸은 '실수'가 아니라 **주문이 아직 끝나지 않았다는 뜻**입니다.

그래서 이 분석에서는 `dropna()` 로 빈 칸 있는 행을 지우는 대신, **`order_status == "delivered"` (배송 완료 주문)** 를 분석 대상으로 정합니다.
`dropna()` 를 쓰면 배송 완료인데 승인일만 빠진 주문까지 함께 지워지고, 반대로 취소됐는데 날짜가 다 있는 주문은 남는 등 기준이 모호해집니다.

</div>

In [9]:
# 데이터 기간 확인 — 문자열을 날짜형(datetime)으로 바꿔야 최솟값 · 최댓값을 날짜로 비교할 수 있습니다.
purchase = pd.to_datetime(orders["order_purchase_timestamp"])
print("첫 주문:", purchase.min())
print("마지막 주문:", purchase.max())

# 월별 주문 수의 마지막 몇 달 — 2018년 9월 이후는 주문이 거의 없음(데이터 수집이 끝난 시점)
purchase.dt.to_period("M").value_counts().sort_index().tail(5)

첫 주문: 2016-09-04 21:15:19
마지막 주문: 2018-10-17 17:30:18


order_purchase_timestamp
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**데이터의 끝을 조심하세요.** 2016년 9~12월과 2018년 9~10월은 주문이 극히 적습니다. 실제로 장사가 안 된 것이 아니라 **데이터가 덜 모인 기간**입니다.
월평균 · 연도 비교를 할 때 이런 기간을 그대로 넣으면 결론이 왜곡됩니다(02 노트북에서 직접 확인).

</div>

In [10]:
# 나머지 표도 간단히 훑어보기 — 가격(price)과 배송비(freight_value)의 분포
items[["price", "freight_value"]].describe()

,price,freight_value
count,"112,650.00","112,650.00"
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,"6,735.00",409.68


---
# PART B. 고객 분석

## B-1. 고객은 몇 명일까? (Q1)

In [11]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [12]:
# nunique(): 서로 다른 값의 개수
print("customer_id 개수       :", customers["customer_id"].nunique())          # 주문마다 새로 생기는 번호
print("customer_unique_id 개수:", customers["customer_unique_id"].nunique())   # 실제 사람 수

customer_id 개수       : 99441
customer_unique_id 개수: 96096


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**실제 고객 수는 `customer_unique_id` 기준 96,096명**입니다. `customer_id` 는 주문할 때마다 새로 만들어지므로 주문 수(99,441)와 같습니다.

</div>

## B-2. 다시 사는 고객은 얼마나 될까? (재구매율)

고객 1명이 주문을 몇 번 했는지 세려면 orders(주문)와 customers(실제 사람 번호)를 **`customer_id` 로 합친(merge)** 뒤 사람별로 묶습니다.

In [13]:
# merge(왼쪽 표, 오른쪽 표, on=연결 열): 같은 customer_id 를 가진 행끼리 옆으로 붙임
order_customer = pd.merge(orders, customers, on="customer_id")

# 사람(customer_unique_id)별 주문 수
orders_per_person = order_customer.groupby("customer_unique_id")["order_id"].nunique()

# 주문 횟수별 고객 수: 1번 주문한 사람, 2번 주문한 사람 …
orders_per_person.value_counts().sort_index().head(6)

order_id
1    93099
2     2745
3      203
4       30
5        8
6        6
Name: count, dtype: int64

In [14]:
repeat_rate = (orders_per_person > 1).mean() * 100     # True(2번 이상)의 비율 = 재구매율
print(f"재구매 고객: {(orders_per_person > 1).sum():,}명  /  재구매율: {repeat_rate:.1f}%")

재구매 고객: 2,997명  /  재구매율: 3.1%


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 1 — 재구매율이 약 3%로 매우 낮습니다.** 고객 대부분이 한 번 사고 떠납니다.
신규 고객을 데려오는 비용이 크다는 점을 생각하면 **재구매를 늘리는 것(쿠폰, 추천, 배송 만족도 개선)** 이 이 회사의 중요한 과제입니다. 02 노트북의 리뷰 분석이 그 단서를 줍니다.

</div>

## B-3. 고객은 어디에 살까? (Q2)

브라질은 27개 주(state)로 나뉩니다. 먼저 **주 단위**로 큰 그림을 보고, 다음에 도시 단위로 자세히 봅니다.

In [15]:
# 같은 사람이 여러 번 나오지 않도록 실제 고객 기준으로 한 번씩만 남김
unique_customers = customers.drop_duplicates("customer_unique_id")

state_count = unique_customers["customer_state"].value_counts()      # 주별 고객 수 (많은 순)
state_share = (state_count / state_count.sum() * 100).round(1)       # 비율(%)
pd.DataFrame({"고객 수": state_count, "비율(%)": state_share}).head(10)

,고객 수,비율(%)
customer_state,,
SP,40295,41.90
RJ,12377,12.90
MG,11255,11.70
RS,5277,5.50
PR,4882,5.10
SC,3529,3.70
BA,3276,3.40
DF,2073,2.20
ES,1963,2.00


In [16]:
fig = px.bar(
    x=state_count.index, y=state_count.values,
    labels={"x": "주(state)", "y": "고객 수"},
    title="주(state)별 고객 수 — 상파울루(SP) 한 곳에 약 42%",
)
fig.show()

**메서드 체이닝** — 아래처럼 `groupby → nunique → sort_values` 를 점(.)으로 이어 쓰는 방법입니다.
여러 줄로 나눠 쓰고 싶으면 전체를 **소괄호 ( )** 로 감싸면 됩니다(줄 끝에 `\` 를 붙이는 방법보다 실수가 적습니다).

In [17]:
city_count = (
    customers.groupby("customer_city")["customer_unique_id"]   # 도시별로 묶어 고객 번호 열 선택
    .nunique()                                                 # 도시마다 실제 고객 수
    .sort_values(ascending=False)                              # 많은 순서로 정렬
)
top20_city = city_count.head(20)

fig = px.bar(x=top20_city.index, y=top20_city.values,
             labels={"x": "도시", "y": "고객 수"}, title="고객이 많은 도시 TOP 20")
fig.update_layout(xaxis_tickangle=-45)     # 도시 이름이 겹치지 않게 45도 기울이기
fig.show()

In [18]:
# enumerate(): 순서 번호(0, 1, 2 …)와 값을 함께 꺼냄 → 순위표 출력
for rank, city in enumerate(top20_city.index[:5], start=1):
    print(f"TOP {rank}: {city}  ({top20_city[city]:,}명)")

TOP 1: sao paulo  (14,984명)
TOP 2: rio de janeiro  (6,620명)
TOP 3: belo horizonte  (2,672명)
TOP 4: brasilia  (2,069명)
TOP 5: curitiba  (1,465명)


## B-4. 고객은 어떤 결제 수단을 쓸까? (Q3)

In [19]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [20]:
print(payments["payment_type"].value_counts())    # 결제 '행' 기준 개수

# not_defined(정의되지 않음)는 3건뿐인 오류 데이터 → 제외
payments = payments[payments["payment_type"] != "not_defined"]

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


결제 행 수는 **주문 수가 아닙니다.** 한 주문을 신용카드 + 바우처처럼 여러 수단으로 나눠 낸 경우가 있기 때문입니다.

In [21]:
# 주문 1건에 쓰인 결제 수단 종류 수
payments.groupby("order_id")["payment_type"].nunique().value_counts()

payment_type
1    97191
2     2246
Name: count, dtype: int64

결제 수단을 2가지 쓴 주문이 약 2,200건 있습니다. 주문을 결제 수단별로 그냥 세면 이 주문들이 **두 번** 세어집니다.
그래서 주문마다 **가장 많은 금액을 낸 수단 하나(주 결제 수단)** 를 골라 비율을 구합니다.

In [22]:
main_payment = (
    payments.sort_values("payment_value", ascending=False)   # 금액이 큰 결제가 위로 오게 정렬
    .drop_duplicates("order_id")                              # 주문마다 맨 위(가장 큰 결제) 하나만 남김
)
main_type_count = main_payment["payment_type"].value_counts()
main_type_count

payment_type
credit_card    74975
boleto         19784
voucher         3151
debit_card      1527
Name: count, dtype: int64

In [23]:
fig = go.Figure(go.Pie(
    labels=main_type_count.index,
    values=main_type_count.values,
    textinfo="label+percent",          # 조각에 이름과 비율 표시
    hole=0.4,                          # 가운데 구멍(도넛 모양) — 비율 비교가 더 쉬움
))
fig.update_layout(title=dict(text="주 결제 수단 비율 (주문 기준)", x=0.5))
fig.show()

In [24]:
# 신용카드 할부 개월 수(payment_installments) 분포
card = payments[payments["payment_type"] == "credit_card"]
print("평균 할부 개월:", round(card["payment_installments"].mean(), 2), "/ 일시불(1개월) 비율:",
      round((card["payment_installments"] == 1).mean() * 100, 1), "%")

fig = px.histogram(card, x="payment_installments", nbins=24,
                   labels={"payment_installments": "할부 개월 수"}, title="신용카드 할부 개월 수 분포")
fig.show()

평균 할부 개월: 3.51 / 일시불(1개월) 비율: 33.1 %


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 2 — 신용카드가 주문의 약 75%, 브라질 고유 결제 방식인 볼레투(boleto, 은행 고지서 결제)가 약 20%** 입니다.
신용카드 결제의 3분의 2가 **할부**라는 점도 특징입니다(일시불은 약 33%). 할부 혜택이 구매 결정에 중요한 시장이라는 뜻입니다.

</div>

---
# PART C. 매출 분석 (Q4)

## C-1. 분석 대상 정하기 — '매출'은 어떤 주문의 금액인가?

1. **배송 완료(delivered) 주문만** 매출로 봅니다(취소 · 품절 · 배송 중 제외).
2. 주문 금액 = 그 주문의 **결제 금액(payment_value) 합계** (결제를 여러 번 했으면 더함).

In [25]:
# ① 배송 완료 주문만 고르기 — copy(): 원본과 분리된 복사본을 만들어 안전하게 수정
delivered = orders[orders["order_status"] == "delivered"].copy()
print("배송 완료 주문:", len(delivered), "건")

# ② 주문별 결제 금액 합계 — payment_value 열 '하나만' 골라서 더하는 것이 중요!
order_value = payments.groupby("order_id", as_index=False)["payment_value"].sum()
order_value.head()

배송 완료 주문: 96478 건


,order_id,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**흔한 실수** — `payments.groupby("order_id").sum()` 처럼 열을 고르지 않고 더하면 **모든 열**을 더합니다.
숫자 열(할부 개월, 결제 순번)은 의미 없는 합계가 되고, 문자 열(payment_type)은 `"vouchercredit_card"` 처럼 **글자가 이어 붙습니다.** 필요한 열만 골라서 계산하세요.

</div>

In [26]:
# ③ 주문 + 주문 금액 합치기. 이제 한 행 = 주문 1건이 그대로 유지됩니다.
sales = pd.merge(delivered, order_value, on="order_id", how="inner")   # inner: 양쪽에 모두 있는 주문만
print("행 수:", len(sales), " / order_id 중복 없음?", sales["order_id"].is_unique)

행 수: 96477  / order_id 중복 없음? True


## C-2. 날짜 다루기 — `pd.to_datetime`

CSV 에서 읽은 날짜는 그냥 **글자(str)** 입니다. 날짜형(datetime64)으로 바꿔야 월별로 묶거나 날짜끼리 뺄 수 있습니다.

| 형식 기호 | 의미 | 예 |
|---|---|---|
| `%Y` / `%y` | 연도 4자리 / 2자리 | 2018 / 18 |
| `%m` · `%d` | 월 · 일 (두 자리) | 01~12 · 01~31 |
| `%H` · `%M` · `%S` | 시(24시간) · 분 · 초 | 00~23 · 00~59 · 00~59 |

`errors=` 옵션 — `"raise"`(기본, 바꿀 수 없는 값이 있으면 오류) · `"coerce"`(바꿀 수 없으면 빈 값 NaT 로)

In [27]:
sales["order_purchase_timestamp"] = pd.to_datetime(
    sales["order_purchase_timestamp"],
    format="%Y-%m-%d %H:%M:%S",      # '2017-10-02 10:56:33' 모양이라고 알려 주면 더 빠르고 정확
    errors="raise",
)
sales[["order_purchase_timestamp", "payment_value"]].dtypes

order_purchase_timestamp    datetime64[us]
payment_value                      float64
dtype: object

## C-3. 주문 1건의 평균 금액 — 객단가(AOV)

**객단가(AOV, Average Order Value)** = 매출 ÷ 주문 수. 쇼핑몰의 가장 기본적인 지표입니다.

In [28]:
sales["payment_value"].describe()

count   96,477.00
mean       159.86
std        218.81
min          9.59
25%         61.88
50%        105.28
75%        176.33
max     13,664.08
Name: payment_value, dtype: float64

In [29]:
aov_mean = sales["payment_value"].mean()
aov_median = sales["payment_value"].median()
print(f"평균 객단가: R$ {aov_mean:,.2f}   /   중앙값: R$ {aov_median:,.2f}")

# 대부분 주문은 작은 금액이고, 아주 비싼 주문 몇 개가 평균을 끌어올림 → 오른쪽으로 꼬리가 긴 분포
fig = px.histogram(sales[sales["payment_value"] <= 1000], x="payment_value", nbins=50,
                   labels={"payment_value": "주문 금액 (R$, 1,000 이하만 표시)"}, title="주문 금액 분포")
fig.add_vline(x=aov_median, line_dash="dash", line_color="green", annotation_text="중앙값")
fig.add_vline(x=aov_mean, line_dash="dash", line_color="red", annotation_text="평균")
fig.show()

평균 객단가: R$ 159.86   /   중앙값: R$ 105.28


<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 3 — 평균 객단가는 약 160헤알 이지만 '보통 주문'(중앙값)은 약 105헤알** 입니다.
최대 13,664헤알 같은 큰 주문이 평균을 끌어올리기 때문입니다. 금액처럼 한쪽으로 치우친 데이터는 **평균과 중앙값을 함께** 보고하세요.

</div>

## C-4. 월별 매출 — `pd.Grouper`

`groupby(pd.Grouper(key="날짜열", freq="MS"))` 는 날짜 열을 **한 달 단위**로 묶습니다.

| freq | 묶는 단위 | 결과에 붙는 날짜 |
|---|---|---|
| `"D"` · `"W"` | 일 · 주 | |
| `"MS"` | 월 (**M**onth **S**tart) | 그 달의 **1일** (2017-11-01) |
| `"ME"` | 월 (**M**onth **E**nd) | 그 달의 **마지막 날** (2017-11-30) — pandas 2.2 부터 `"M"` 대신 사용 |
| `"QS"` / `"QE"` · `"YS"` / `"YE"` | 분기 · 연 | 시작일 / 마지막 날 |

그래프를 그릴 때는 **`"MS"`(월초)** 를 권장합니다. `"ME"` 로 묶으면 11월 매출이 11월 30일에 찍혀 x축의 **'12월' 눈금 바로 옆**에 그려지는 착시가 생깁니다.

In [30]:
monthly = (
    sales.groupby(pd.Grouper(key="order_purchase_timestamp", freq="MS"))      # MS: 각 달을 1일 날짜로 표시
    .agg(revenue=("payment_value", "sum"),       # 매출 합계
         orders=("order_id", "count"),            # 주문 수 (이제 한 행 = 주문 1건이므로 count 가 정확)
         aov=("payment_value", "mean"))           # 그 달의 객단가
)
monthly

,revenue,orders,aov
order_purchase_timestamp,,,
2016-10-01,"46,566.71",265,175.72
2016-11-01,0.00,0,NaN
2016-12-01,19.62,1,19.62
2017-01-01,"127,545.67",750,170.06
2017-02-01,"271,298.65",1653,164.13
2017-03-01,"414,369.39",2546,162.75
2017-04-01,"390,952.18",2303,169.76
2017-05-01,"567,066.73",3546,159.92
2017-06-01,"490,225.60",3135,156.37


2016년 10~12월은 주문이 거의 없습니다(데이터 수집 초기). **월평균 매출은 데이터가 온전한 2017년 1월 ~ 2018년 8월(20개월)** 로 계산합니다.

In [31]:
full = monthly.loc["2017-01":"2018-08"]          # 날짜 인덱스는 '연-월' 글자로 범위를 자를 수 있음
print("월평균 매출(2017.01~2018.08):", f"R$ {full['revenue'].mean():,.0f}")
print("전체 기간 총매출             :", f"R$ {sales['payment_value'].sum():,.0f}")

best_month = full["revenue"].idxmax()            # idxmax(): 가장 큰 값이 있는 '인덱스(날짜)'
print("최대 매출 월:", best_month.strftime("%Y-%m"), f"R$ {full['revenue'].max():,.0f}")

월평균 매출(2017.01~2018.08): R$ 768,794
전체 기간 총매출             : R$ 15,422,462
최대 매출 월: 2017-11 R$ 1,153,528


In [32]:
# 최대 매출 월의 일별 주문 수 — 무슨 일이 있었을까?
nov = sales[sales["order_purchase_timestamp"].dt.strftime("%Y-%m") == "2017-11"]
nov.groupby(nov["order_purchase_timestamp"].dt.date)["order_id"].count().sort_values(ascending=False).head(3)

order_purchase_timestamp
2017-11-24    1147
2017-11-25     487
2017-11-27     395
Name: order_id, dtype: int64

<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 4 — 2017년 11월 24일(블랙프라이데이) 하루에만 1,100건 넘는 주문**이 들어왔습니다. 11월이 최대 매출 월이 된 이유입니다.
대형 할인 행사에 맞춘 재고 · 물류 준비가 중요하다는 뜻입니다(02 노트북에서 이때 배송이 늦어진 것도 확인합니다).

</div>

## C-5. 보고서용 그래프 만들기

기본 그래프를 다음 순서로 다듬습니다: **① 불완전한 기간 제외 → ② 값 표시 → ③ 최대 월만 다른 색 → ④ 주석 달기 → ⑤ 제목 · 축 정리**.
최대 월의 위치와 값은 직접 적지 않고 **코드로 계산**합니다. 그래야 데이터가 바뀌어도 그래프가 맞게 그려집니다.

In [33]:
# 막대 색: 최대 매출 월만 강조색, 나머지는 기본색 (리스트 컴프리헨션)
colors = ["#F24472" if m == best_month else "#03588C" for m in full.index]

fig = go.Figure(go.Bar(
    x=full.index, y=full["revenue"],
    marker_color=colors,
    texttemplate="%{y:,.0f}",          # 막대 위 숫자: 천 단위 쉼표, 소수점 없음
    textposition="outside",
    textfont_size=8,
))
fig.add_annotation(                     # 최대 월에 말풍선 달기
    x=best_month, y=full["revenue"].max(),
    text="<b>블랙프라이데이</b>", showarrow=True, arrowhead=2, ax=0, ay=-45,
    bgcolor="#F25D50", font=dict(color="white", size=11),
)
fig.update_layout(
    title=dict(text="<b>Olist 월별 매출</b> (배송 완료 주문, 2017.01 ~ 2018.08)", x=0.5),
    xaxis=dict(title="주문 월", dtick="M1", tickformat="%Y-%m", tickangle=-45),   # 한 달마다 눈금
    yaxis=dict(title="매출 (R$)"),
    margin=dict(t=80),
)
fig.show()

## C-6. 매출은 왜 늘었을까? — 주문 수 vs 객단가

매출 = **주문 수 × 객단가** 입니다. 둘 중 무엇이 늘었는지 나눠 보면 성장의 원인을 알 수 있습니다.

In [34]:
from plotly.subplots import make_subplots     # 그래프 여러 개를 한 그림에 배치

fig = make_subplots(rows=1, cols=2, subplot_titles=("월별 주문 수", "월별 객단가 (R$)"))
fig.add_trace(go.Bar(x=full.index, y=full["orders"], name="주문 수"), row=1, col=1)
fig.add_trace(go.Scatter(x=full.index, y=full["aov"], mode="lines+markers", name="객단가"), row=1, col=2)
fig.update_yaxes(range=[0, 200], row=1, col=2)   # 객단가 축을 0부터 — 변화가 과장되어 보이지 않게
fig.update_layout(showlegend=False, height=380)
fig.show()

<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**인사이트 5 — 성장은 '주문 수' 덕분입니다.** 월 주문 수는 2017년 초 수백 건에서 2018년 6,000~7,000건으로 늘었지만, 객단가는 145~170헤알 사이에서 거의 그대로입니다.
즉, 더 비싸게 판 것이 아니라 **더 많은 고객이 산 것**입니다.

</div>

---
# 정리

| 질문 | 답 |
|---|---|
| Q1 고객 수 · 재구매 | 실제 고객 96,096명, **재구매율 약 3%** |
| Q2 지역 | **상파울루(SP) 약 42%**, RJ · MG 까지 3개 주가 약 66% |
| Q3 결제 수단 | 신용카드 약 75%(그중 약 3분의 2가 할부), 볼레투 약 20% |
| Q4 매출 | 객단가 평균 160헤알 / 중앙값 105헤알, 월평균 매출 약 77만헤알, 최대는 2017년 11월(블랙프라이데이) |

**이번에 익힌 pandas** : `read_csv` · `info` · `value_counts` · `nunique` · `is_unique` · `crosstab` · `merge` · `groupby` + `agg` · `drop_duplicates` · `to_datetime` · `pd.Grouper` · `idxmax`

# 연습문제

1. `sellers` 표에서 판매자가 가장 많은 주(`seller_state`) 3곳과 판매자 수를 구하세요.
2. 결제 수단별 **평균 결제 금액**을 구하세요(`payments` 사용). 어떤 수단의 평균 금액이 가장 클까요?
3. `sales` 에서 **2018년 1~8월** 만 골라 월별 객단가의 최솟값과 최댓값을 구하세요.

In [35]:
# 1번 정답 예시
tables["sellers"]["seller_state"].value_counts().head(3)

seller_state
SP    1849
PR     349
MG     244
Name: count, dtype: int64

In [36]:
# 2번 정답 예시
payments.groupby("payment_type")["payment_value"].mean().sort_values(ascending=False)

payment_type
credit_card   163.32
boleto        145.03
debit_card    142.57
voucher        65.70
Name: payment_value, dtype: float64

In [37]:
# 3번 정답 예시
m2018 = monthly.loc["2018-01":"2018-08", "aov"]
print("최솟값:", round(m2018.min(), 2), "/ 최댓값:", round(m2018.max(), 2))

최솟값: 147.45 / 최댓값: 167.26
